# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [6]:
df = pd.read_csv("AviationData.csv", encoding="latin1", low_memory=False)

df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [11]:
df.shape

(88889, 31)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [9]:
df.isnull().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Investigation.Type            0
Event.Date                    0
Accident.Number               0
Event.Id                      0
dtype: i

In [10]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [13]:
#converting date column
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

In [14]:
#creating year column
df["year"] = df["Event.Date"].dt.year

In [15]:
#filtering aircraft by year
df = df[df["year"] >= 1983]

In [16]:
#need to see the shape
df.shape

(85289, 32)

In [18]:
relevant_cols = [
    "Event.Date",
    "year",
    "Make",
    "Model",
    "Aircraft.Category",
    "Aircraft.damage",
    "Number.of.Engines",
    "Engine.Type",
    "Purpose.of.flight",
    "Weather.Condition",
    "Broad.phase.of.flight",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

df = df[relevant_cols]

df.head()

,Event.Date,year,Make,Model,Aircraft.Category,Aircraft.damage,Number.of.Engines,Engine.Type,Purpose.of.flight,Weather.Condition,Broad.phase.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
3600,1983-01-01,1983,Piccard,AX-6,NaN,NaN,NaN,Unknown,Personal,VMC,Landing,0.0,1.0,0.0,1.0
3601,1983-01-01,1983,Cessna,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,1.0,3.0
3602,1983-01-01,1983,Cessna,182RG,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Landing,0.0,0.0,0.0,2.0
3603,1983-01-01,1983,Cessna,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Takeoff,0.0,0.0,0.0,1.0
3604,1983-01-01,1983,Piper,PA-28R-200,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,2.0,0.0


In [19]:
df.shape

(85289, 15)

In [20]:
df.head()

,Event.Date,year,Make,Model,Aircraft.Category,Aircraft.damage,Number.of.Engines,Engine.Type,Purpose.of.flight,Weather.Condition,Broad.phase.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
3600,1983-01-01,1983,Piccard,AX-6,NaN,NaN,NaN,Unknown,Personal,VMC,Landing,0.0,1.0,0.0,1.0
3601,1983-01-01,1983,Cessna,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,1.0,3.0
3602,1983-01-01,1983,Cessna,182RG,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Landing,0.0,0.0,0.0,2.0
3603,1983-01-01,1983,Cessna,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Takeoff,0.0,0.0,0.0,1.0
3604,1983-01-01,1983,Piper,PA-28R-200,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,2.0,0.0


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [28]:
#ensuring that columns are usable
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

df[injury_cols] = df[injury_cols].fillna(0)
df[injury_cols].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
3600,0.0,1.0,0.0,1.0
3601,0.0,0.0,1.0,3.0
3602,0.0,0.0,0.0,2.0
3603,0.0,0.0,0.0,1.0
3604,0.0,0.0,2.0,0.0


In [30]:
#estimating total number of passengers on each flight
df["Total.People"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"] +
    df["Total.Minor.Injuries"] +
    df["Total.Uninjured"]
)
df["Total.People"].head()

3600    2.0
3601    4.0
3602    2.0
3603    1.0
3604    2.0
Name: Total.People, dtype: float64

In [31]:
#combining fatal and serious injuries
df["Serious.Fatal.Injuries"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"]
)
df["Serious.Fatal.Injuries"].head()

3600    1.0
3601    0.0
3602    0.0
3603    0.0
3604    0.0
Name: Serious.Fatal.Injuries, dtype: float64

In [32]:
#the metric
df["Serious.Fatal.Rate"] = df["Serious.Fatal.Injuries"] / df["Total.People"]
df["Serious.Fatal.Rate"].head()

3600    0.5
3601    0.0
3602    0.0
3603    0.0
3604    0.0
Name: Serious.Fatal.Rate, dtype: float64

In [34]:
#removing invalid rowes
df = df[df["Total.People"] > 0]
df.head()

,Event.Date,year,Make,Model,Aircraft.Category,Aircraft.damage,Number.of.Engines,Engine.Type,Purpose.of.flight,Weather.Condition,Broad.phase.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.People,Serious.Fatal.Injuries,Serious.Fatal.Rate
3600,1983-01-01,1983,Piccard,AX-6,NaN,NaN,NaN,Unknown,Personal,VMC,Landing,0.0,1.0,0.0,1.0,2.0,1.0,0.5
3601,1983-01-01,1983,Cessna,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,1.0,3.0,4.0,0.0,0.0
3602,1983-01-01,1983,Cessna,182RG,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Landing,0.0,0.0,0.0,2.0,2.0,0.0,0.0
3603,1983-01-01,1983,Cessna,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Takeoff,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3604,1983-01-01,1983,Piper,PA-28R-200,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,2.0,0.0,2.0,0.0,0.0


In [35]:
#results
df[[
    "Total.People",
    "Serious.Fatal.Injuries",
    "Serious.Fatal.Rate"
]].head()

,Total.People,Serious.Fatal.Injuries,Serious.Fatal.Rate
3600,2.0,1.0,0.5
3601,4.0,0.0,0.0
3602,2.0,0.0,0.0
3603,1.0,0.0,0.0
3604,2.0,0.0,0.0


Fatal and serious injury risk was calculated by first estimating the total number of individuals onboard each flight as the sum of fatal, serious, minor injuries, and uninjured passengers. A normalized metric was then constructed by dividing the number of fatal and serious injuries by this total, producing a comparable injury rate across aircraft of different sizes.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [36]:
#inspecting the column
df["Aircraft.damage"].value_counts()

Aircraft.damage
Substantial    61506
Destroyed      17479
Minor           2369
Unknown           92
Name: count, dtype: int64

In [38]:
#standardize the column
df["Aircraft.damage"] = df["Aircraft.damage"].str.strip().str.title()
df["Aircraft.damage"]

3600             NaN
3601     Substantial
3602     Substantial
3603     Substantial
3604     Substantial
            ...     
88882            NaN
88883            NaN
88884            NaN
88886    Substantial
88888            NaN
Name: Aircraft.damage, Length: 83980, dtype: object

The aircraft damage column was standardized by stripping whitespace and normalizing text capitalization to ensure consistent categorization of damage levels.

In [40]:
#derived/created column 
df["Destroyed.Flag"] = df["Aircraft.damage"].apply(
    lambda x: 1 if x == "Destroyed" else 0
)
df[["Aircraft.damage", "Destroyed.Flag"]].head()

,Aircraft.damage,Destroyed.Flag
3600,NaN,0
3601,Substantial,0
3602,Substantial,0
3603,Substantial,0
3604,Substantial,0


In [41]:
df["Destroyed.Flag"].value_counts()

Destroyed.Flag
0    66501
1    17479
Name: count, dtype: int64

I created the binary variable (Destroyed.Flag) as a representation of whether an aircraft was destroyed (1) or not (0), effectively simplifying damage severity into a measurable outcome aligned with the client’s interest in aircraft robustness.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [42]:
#investigating colunm MAKE
df["Make"].value_counts(dropna=False).head(20)

Make
Cessna               20874
Piper                11288
CESSNA                4838
Beech                 4057
PIPER                 2813
Bell                  2005
Boeing                1392
Mooney                1036
BEECH                 1025
Grumman                990
Robinson               920
Bellanca               824
Hughes                 741
BOEING                 705
Schweizer              610
Air Tractor            577
BELL                   569
Mcdonnell Douglas      497
Aeronca                457
Maule                  427
Name: count, dtype: int64

In [43]:
#Checking uniqueness
df["Make"].nunique()

8091

To achieve consistency within Make Column. I'll do the following:
Text Standardization to Convert all values to uppercase to eliminate case inconsistencies.
Removing whitespace stripping leading and trailing spaces
Handling missing/unknown values: Ensured missing values were labeled consistently
Filtering low-frequency manufacturers: Removed manufacturers with fewer than 50 records to improve statistical reliability and reduce noise during analysis

In [48]:
#Standardization of text
df["Make"] = df["Make"].str.strip().str.upper()


In [49]:
#Checking results
df["Make"].value_counts().head(20)

Make
CESSNA               25712
PIPER                14101
BEECH                 5082
BELL                  2574
BOEING                2097
MOONEY                1275
ROBINSON              1185
GRUMMAN               1068
BELLANCA               982
HUGHES                 878
SCHWEIZER              753
AIR TRACTOR            668
AERONCA                606
MAULE                  571
MCDONNELL DOUGLAS      560
CHAMPION               507
STINSON                421
AERO COMMANDER         411
DE HAVILLAND           400
LUSCOMBE               391
Name: count, dtype: int64

In [51]:
#Filtering to critical manufacturers
make_counts = df["Make"].value_counts()

valid_makes = make_counts[make_counts >= 50].index

df = df[df["Make"].isin(valid_makes)]

In [52]:
#View the column
df["Make"].value_counts()

Make
CESSNA                         25712
PIPER                          14101
BEECH                           5082
BELL                            2574
BOEING                          2097
                               ...  
LANCAIR                           52
SMITH, TED AEROSTAR               51
FLIGHT DESIGN GMBH                50
GRUMMAN AMERICAN AVN. CORP.       50
BOEING STEARMAN                   50
Name: count, Length: 97, dtype: int64

In [53]:
#view shape
df.shape

(70178, 19)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [57]:
#inspect Model column
df["Model"].isnull().sum()


16

In [58]:
#further inspection
df["Model"].value_counts().head(20)

Model
152          2226
172          1641
172N         1093
PA-28-140     868
172M          759
150           748
172P          663
182           615
180           596
150M          553
PA-18-150     550
PA-18         548
PA-28-180     548
PA-28-161     526
PA-28-181     508
206B          486
150L          436
A36           428
PA-38-112     425
G-164A        423
Name: count, dtype: int64

In [60]:
#handling NaNs gracifuly
df = df[df["Model"].notna()]
df

,Event.Date,year,Make,Model,Aircraft.Category,Aircraft.damage,Number.of.Engines,Engine.Type,Purpose.of.flight,Weather.Condition,Broad.phase.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.People,Serious.Fatal.Injuries,Serious.Fatal.Rate,Destroyed.Flag
3601,1983-01-01,1983,CESSNA,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,1.0,3.0,4.0,0.0,0.0,0
3602,1983-01-01,1983,CESSNA,182RG,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Landing,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0
3603,1983-01-01,1983,CESSNA,182P,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Takeoff,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0
3604,1983-01-01,1983,PIPER,PA-28R-200,NaN,Substantial,1.0,Reciprocating,Personal,VMC,Approach,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0
3605,1983-01-01,1983,CESSNA,140,NaN,Substantial,1.0,Reciprocating,Instructional,VMC,Landing,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88882,2022-12-21,2022,GRUMMAN AMERICAN AVN. CORP.,AA-5B,NaN,NaN,NaN,NaN,Instructional,NaN,NaN,0.0,1.0,0.0,1.0,2.0,1.0,0.5,0
88883,2022-12-22,2022,AIR TRACTOR,AT502,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0
88884,2022-12-26,2022,PIPER,PA-28-151,NaN,NaN,NaN,NaN,Personal,NaN,NaN,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0
88886,2022-12-26,2022,AMERICAN CHAMPION AIRCRAFT,8GCBC,Airplane,Substantial,1.0,NaN,Personal,VMC,NaN,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0


Rows with missing aircraft model information were removed, as model-level identification is essential for aircraft-specific safety analysis.

In [61]:
#checking if models are unique
df.groupby("Model")["Make"].nunique().sort_values(ascending=False).head(10)

Model
G-164A    7
500       7
G-164B    6
AA-5B     5
G-164C    5
AA-5      5
G-164     5
500-S     4
AA-1B     4
AA-1C     4
Name: Make, dtype: int64

Model is not unique. i.e same model name<> same aircraft. This means that by using Model only, i run the risk of mixing different aircraft type and produce invalid manufacturer comparisons

In [62]:
#creating a unique aircraft identifier
df["Aircraft.Type"] = df["Make"] + " " + df["Model"]

In [63]:
#verifying the created column
df["Aircraft.Type"].value_counts().head(10)

Aircraft.Type
CESSNA 152         2226
CESSNA 172         1641
CESSNA 172N        1093
PIPER PA-28-140     868
CESSNA 172M         759
CESSNA 150          748
CESSNA 172P         663
CESSNA 182          615
CESSNA 180          595
CESSNA 150M         553
Name: count, dtype: int64

Analysis of the Model column showed that aircraft model names are not unique across manufacturers, making Model column inadequate as a standalone identifier.
I created a new column (Aircraft.Type) by combining Make and Model to ensure each aircraft type is uniquely identified.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [64]:
#inspecting engine.type column
df["Engine.Type"].value_counts(dropna=False)

Engine.Type
Reciprocating      55591
NaN                 4717
Turbo Shaft         2991
Turbo Prop          2788
Turbo Fan           2053
Unknown             1478
Turbo Jet            540
Geared Turbofan        1
LR                     1
UNK                    1
NONE                   1
Name: count, dtype: int64

The Engine.Type column contained inconsistent labels and missing values. I clean this by:
Standardizing all text to lowercase for consistency
Converting known missing labels (NaN, unknown, unk, none) into a single category: "unknown"

In [65]:
#text standardization
df["Engine.Type"] = df["Engine.Type"].str.lower()

In [70]:
#Unifying NaN, unknown, unk, none into "unknown"
df["Engine.Type"] = df["Engine.Type"].replace(
    ["NaN", "unknown", "unk", "none"], "unknown"
)
df["Engine.Type"] = df["Engine.Type"].fillna("unknown")
df["Engine.Type"].value_counts()

Engine.Type
reciprocating      55591
unknown             6197
turbo shaft         2991
turbo prop          2788
turbo fan           2053
turbo jet            540
geared turbofan        1
lr                     1
Name: count, dtype: int64

In [71]:
#inspecting weather.condition
df["Weather.Condition"].value_counts(dropna=False)

Weather.Condition
VMC    61153
IMC     5271
NaN     2847
UNK      683
Unk      208
Name: count, dtype: int64

I consolidated (NaN, UNK, Unk) into a single category "UNK". This ensures consistency in downstream analysis of weather-related risk factors.

In [72]:
#Standardizing case
df["Weather.Condition"] = df["Weather.Condition"].str.upper()

In [73]:
#Unifying NaN, UNK, Unk into "UNK"
df["Weather.Condition"] = df["Weather.Condition"].fillna("UNK")
df["Weather.Condition"] = df["Weather.Condition"].replace(["UNK", "UNK "], "UNK")
df["Weather.Condition"].value_counts(dropna=False)

Weather.Condition
VMC    61153
IMC     5271
UNK     3738
Name: count, dtype: int64

In [74]:
#Inspecting number.of.engines
df["Number.of.Engines"].value_counts(dropna=False)

Number.of.Engines
1.0    55027
2.0     9443
NaN     4163
0.0      722
3.0      427
4.0      379
8.0        1
Name: count, dtype: int64

Missing values were retained, and zero-engine entries were kept as-is. Column retained in numeric form.

In [76]:
#Ensuring numeric data type
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")
df["Number.of.Engines"].value_counts(dropna=False)

Number.of.Engines
1.0    55027
2.0     9443
NaN     4163
0.0      722
3.0      427
4.0      379
8.0        1
Name: count, dtype: int64

In [77]:
#inspecting purpose.of.light
df["Purpose.of.flight"].value_counts(dropna=False).head(15)

Purpose.of.flight
Personal               37435
Instructional           9430
Unknown                 5473
NaN                     4437
Aerial Application      4142
Business                3449
Positioning             1449
Other Work Use          1066
Aerial Observation       712
Public Aircraft          652
Ferry                    644
Executive/corporate      429
Skydiving                175
Flight Test              163
Banner Tow                97
Name: count, dtype: int64

In [ ]:
I have only standardized unknown and unified NaN and Unknown into unknown

In [78]:
#standardizing text
df["Purpose.of.flight"] = df["Purpose.of.flight"].str.lower()

In [79]:
#unify NaN and Unknown into unknown
df["Purpose.of.flight"] = df["Purpose.of.flight"].fillna("unknown")
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace("unknown", "unknown")
df["Purpose.of.flight"].value_counts()

Purpose.of.flight
personal                     37435
unknown                       9910
instructional                 9430
aerial application            4142
business                      3449
positioning                   1449
other work use                1066
aerial observation             712
public aircraft                652
ferry                          644
executive/corporate            429
skydiving                      175
flight test                    163
banner tow                      97
external load                   85
public aircraft - federal       76
public aircraft - state         58
public aircraft - local         52
glider tow                      41
firefighting                    33
air race show                   30
air race/show                   18
air drop                         9
pubs                             3
asho                             3
publ                             1
Name: count, dtype: int64

In [80]:
#Inspecting broad.phase.of.flight
df["Broad.phase.of.flight"].value_counts(dropna=False)

Broad.phase.of.flight
NaN            19708
Landing        13103
Takeoff         9869
Cruise          8462
Maneuvering     6347
Approach        5255
Taxi            1678
Climb           1665
Descent         1573
Go-around       1192
Standing         828
Unknown          399
Other             83
Name: count, dtype: int64

I consilidated NaN and "Unknown" into a single "unknown" category. I retained the category labelled "Other" to preserve completeness. Also adopted the standardization of text

In [81]:
#text standardization
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].str.lower()

In [83]:
#unifying NaN and Unknown with "unknown"
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].fillna("unknown")
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].replace("unknown", "unknown")
df["Broad.phase.of.flight"].value_counts(dropna=False)

Broad.phase.of.flight
unknown        20107
landing        13103
takeoff         9869
cruise          8462
maneuvering     6347
approach        5255
taxi            1678
climb           1665
descent         1573
go-around       1192
standing         828
other             83
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [85]:
#checking columns with missing values greater than 70% of its values
threshold = 0.7
missing_pct = df.isna().mean()

cols_to_drop = missing_pct[missing_pct > threshold].index
cols_to_drop


Index(['Aircraft.Category'], dtype='object')

In [86]:
#dropping Aircraft.Category
df = df.drop(columns=["Aircraft.Category"])


In [88]:
#inspecting data
df.shape


(70162, 19)

In [89]:
#inspecting columns
df.columns

Index(['Event.Date', 'year', 'Make', 'Model', 'Aircraft.damage',
       'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight',
       'Weather.Condition', 'Broad.phase.of.flight', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Total.People', 'Serious.Fatal.Injuries', 'Serious.Fatal.Rate',
       'Destroyed.Flag', 'Aircraft.Type'],
      dtype='object')

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [90]:
#choosing file name
output_file = "aviation_cleaned_data.csv"

In [93]:
#save to CSV
df.to_csv(output_file, index=False)

In [94]:
#confirming it is saved
import os

os.path.exists(output_file)

True